In [ ]:
%load_ext autoreload
%autoreload 2
import sys
sys.path.append('..')
import os
from sklearn.datasets import fetch_openml
from src.graph import plot_freq_par_groupe
from src.data_loading import *
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


os.makedirs("../data", exist_ok=True)

if os.path.exists("../data/freq.parquet"):
    freq = pd.read_parquet("../data/freq.parquet")
    sev  = pd.read_parquet("../data/sev.parquet")
else:
    freq = fetch_openml(data_id=41214, as_frame=True).frame
    sev  = fetch_openml(data_id=41215, as_frame=True).frame
    freq.to_parquet("../data/freq.parquet")
    sev.to_parquet("../data/sev.parquet")


print("La première table freq contient les données de " + str(freq.shape[0]) + " polices d'assurance et " + str(freq.shape[1]) + " variables.")
print("La deuxième table sev contient les données de " + str(sev.shape[0]) + " sinistres et " + str(sev.shape[1]) + " variables.")
print(freq.head())
print(sev.head())

### Table `freq` — les contrats (678 013 lignes, 12 variables)
- `IDpol` : identifiant unique de chaque police.
- `ClaimNb` : nombre de sinistres survenus sur la période
- `Exposure` : durée de couverture en années police
- `DrivAge` : âge du conducteur 
- `BonusMalus` : coefficient de Bonus-Malus français, allant de 50 à 350 selon la conduite du conducteur
- `VehPower` : catégorie de puissance du véhicule 
- `VehAge` : âge du véhicule 
- `VehBrand` : marque anonymisée du véhicule
- `VehGas` : type de carburant du véhicule 
- `Density` : densité de la population dans la zone de résidence en hab/km²
- `Area` : discrétisation de `Density`, niveau de densité de la zone de résidence
- `Region` : région administrative française, climat, météo, état des routes ...

### Table `sev` — les sinistres (26 639 lignes, 2 variables)
- `IDpol` : identifiant de la police concernée par le sinistre
- `ClaimAmount` : montant du sinistre, en euros.

Avant de commencer à manipuler les données, il est important de constater s'il y a des erreurs dans les tables de données. Par exemple nous allons regarder s'il y a des doublons dans les polices, des temps d'exposition qui dépassent 1 an, ou s'il y a des fréquences de sinistres anormalement hautes, ce qui traduirait des erreurs de saisie. (Fortement probable sur un jeu de données de cette taille)

In [ ]:
# vérification d'éventuelles valeurs manquantes
print(freq.isnull().sum())

In [ ]:
print(freq.dtypes)
print(sev.dtypes)

In [ ]:
#Vérification d'éventuelles valeurs dupliquées, notamment sur l'identifiant de police
print("le nombre de polices dupliquées est de", freq['IDpol'].duplicated().sum())

In [ ]:
print(freq.groupby('Area')['Density'].describe())

On confirme ici la redondance des features `Area` et `Density`, `Area` étant bien une discrétisation de `Density`. On remarque cependant que pour la zone F, la médiane et le 3ème quartile valent 27000, soit la valeur maximale de `Density`, il s'agit sûrement de zones urbaines très peuplée, comme Paris intra muros par exemple.

In [ ]:
# verification de l'intégrité des données

print(freq[['VehPower','VehAge','DrivAge','BonusMalus','Density']].describe())
print("nombre de score bonus/malus en dehors de la legislation : ", (freq['BonusMalus'] > 350).sum(), (freq['BonusMalus'] < 50).sum())
print("nombre de conducteurs avec un âge inférieur à 18 ans : ", (freq['DrivAge'] < 18).sum())

Les bornes réglementaires de bonus-malus français (50-350) sont respectées et aucun conducteur n'a moins de 18 ans.

Cependant on constate une erreur sur la feature `VehAge` dont le maximum est à 100, ce qui pose question pour l'âge d'une voiture. On va aussi s'intéresser au 100 de DrivAge voir si c'est une valeur normale ou un plafond de la table `freq`. Le reste des variables ne supposent aucun problème.

In [ ]:
print((freq['VehAge'] > 30).sum(), (freq['VehAge'] >= 100).sum())
print((freq['DrivAge'] > 90).sum(), (freq['DrivAge'] >= 100).sum())
print(freq['VehAge'].value_counts().sort_index().tail(15))
print(freq['DrivAge'].value_counts().sort_index().tail(15))


In [ ]:
print(freq['DrivAge'].value_counts().sort_index().loc[92:100])

Le nombre de véhicules à âge élevé décroit normalement jusqu'à 85, puis plus rien jusqu'à 98, et brusquement 48 véhicules entre 99 et 100, impossible, il s'agit surement là d'une valeur code pour signaler un "âge inconnu". De plus tous les autres véhicules au dessus de 30 ans ne sont pas forcément aberrants, mais pour rester dans un seuil raisonnable de travail, nous allons donc les fixer à 30 ans. Sans quoi les effectifs deviennent trop faibles pour estimer quoi que ce soit de fiable.

De même ici la décroissance est régulière et plausible jusqu'à 99 ans ou il se produit un pic anormal, il doit aussi s'agir d'une valeur sentinelle. Nous allons donc plafonner les âges à 90 ans.

In [ ]:
freq['VehAge'] = freq['VehAge'].clip(upper=30)
freq['DrivAge'] = freq['DrivAge'].clip(upper=90)

In [ ]:

print(freq['VehGas'].describe())

#retrait des apostrophes dans la variable VehGas
freq['VehGas'] = freq['VehGas'].str.strip("'")

In [ ]:
print(freq['Exposure'].describe())
print("Nombre de polices dont l'exposition est supérieur à 1 : ", (freq['Exposure'] > 1).sum())

Le fait que certaines exposition dépassent un an (le cas pour 1224 contrats) est une anomalie car on est sur un jeu de données qui est censé contenir des contrats annuels.

On ramène les temps d'exposition des contrats dépassant une durée de 1 an à 1 car il s'agit sûrement là d'une anomalie et de toute façon leur impact sera négligeable. Seulement 1224 sur 678 000 ...

In [ ]:

freq['Exposure'] = freq['Exposure'].clip(upper=1)

On vérifie maintenant s'il n'y a pas des valeurs de `ClaimNb` qui seraient incohérentes.

In [ ]:
print((freq['ClaimNb']).value_counts().sort_index(ascending=False))

La distribution de `ClaimNb` décroît régulièrement jusqu'à 4, puis devient incohérente : 3 à 11 alors qu'il n'y en a pas plus de 2 de 5 à 9. Une loi de probabilité de comptage n'a pas de telle distribution, il s'agit donc sûrement d'un artefact.

In [ ]:
freq.loc[freq['ClaimNb'] >= 4, ['IDpol', 'ClaimNb', 'Exposure']]


Regarder directement les polices ayant un `ClaimNb` de 4 ou plus le confirme, la plupart ont des périodes d'exposition courtes : 16 sinistres sur 4 mois, cela dépasse toute réalité de souscription, un tel client serait sans doute résilié.



J'ai d'abord essayé ici d'appliquer un critère de discrimination sur la fréquence implicite des sinistres par police. En effet un raisonnement simple pourrait être que si on a une fréquence implicite très élevée, il s'agit surement d'une anomalie. Or il est au vu du résultat ci dessous mal adapté car il pénalise les expositions courtes avec peu de sinistres tout en épargnant potentiellement des nombres anormalement grands de sinistre mais sur une longue exposition. 

In [ ]:
extremes = freq.loc[freq['ClaimNb'] >= 4].copy()
extremes['freq_implicite'] = extremes['ClaimNb'] / extremes['Exposure']
extremes[['ClaimNb', 'Exposure', 'freq_implicite']].sort_values('freq_implicite', ascending=False)

Nous allons donc ramener toutes ces valeurs à 4, elles représentent 16 polices sur 678 013 soit 0.002%, l'écart que ce changement produira sur la fréquence annualisée du portefeuille est négligeable.

In [ ]:
freq['ClaimNb']  = freq['ClaimNb'].clip(upper=4)

In [ ]:
print(sev["ClaimAmount"].describe())
print(sev["ClaimAmount"].quantile([0.01, 0.05, 0.95, 0.99]))
print(sev["ClaimAmount"].value_counts().sort_index().head(10))

Une grande majorité des montants de sinistres sont concentrés sur une plage restreinte autour de 1 000 €. Quelques sinistres dépassent les 100 000 euros, et un sinistre exceptionnel dépasse les 4 millions d'euros. Il s'agit d'un sinistre exceptionnel qui fausserait l'estimation de la sévérité, alors que sa survenance n'est pas prédictible à partir des caractéristiques de l'assuré (événement exceptionnel). Un plafonnement suivi d'une réallocation de la charge excédentaire sous forme de chargement mutualisé sera étudié.

On remarque aussi que 1% des sinistres sont inférieurs à 40€, et que certains sinistres sont à 1€, ce qui est douteux. Il s'agit peut-être de sinistres forfaitaires, mais il faudra vérifier si ces sinistres sont bien réels ou s'il s'agit d'erreurs de saisie.

In [ ]:
print((freq['ClaimNb']).value_counts().sort_index())

In [ ]:
#vérification du nombre de sinistre totale de sinistres déclaré et du nombre de ligne dans la table sev
total_claims = freq["ClaimNb"].sum()
print("Le nombre total de sinistres déclarés est de " + str(total_claims) + " et le nombre de lignes dans la table sev est de " + str(sev.shape[0]) + ".")

In [ ]:
# nb_claims_sev regroupe le nombre de sinistres par police dans la table sev, ce qui permet de comparer avec le comptage de sinistres dans la table freq
nb_claims_sev = sev.groupby('IDpol').size().rename('nb_lignes_sev')

#on regroupe ensuite les deux tables freq et nb_claims_sev pour comparer le comptage de sinistres entre les deux tables
comparaison = freq.merge(nb_claims_sev, how='left', left_on='IDpol', right_index=True)

#les polices n'ayant jamais eu de sinistres dans les deux tables auront une valeur NaN dans la colonne nb_lignes_sev, on remplace donc ces valeurs par 0 et on convertit la colonne en int
comparaison['nb_lignes_sev'] = comparaison['nb_lignes_sev'].fillna(0).astype(int)


#on regroupe dans 'différents' les polices pour lesquelles le comptage de sinistres est différent entre freq et sev
differents = comparaison.loc[comparaison['ClaimNb'] != comparaison['nb_lignes_sev']]
print("Nombre de polices pour lesquelles le comptage de sinistres est différent entre freq et sev :", differents.shape[0])
print("Nombre de sinistres déclarés par ces polices : ", differents["ClaimNb"].sum())
print("Nombre de sinistres chiffrés payé par ces polices: ", differents["nb_lignes_sev"].sum())

In [ ]:
#On ajoute dans comparaison une colonne manquant à variable booléenne qui indique si une police a des sinistres déclarés mais auncun de sinistres chiffrés
comparaison["manquant"] = (comparaison["ClaimNb"] > 0) & (comparaison["nb_lignes_sev"] == 0)
print("Nombre de polices ayant déclarés au moins 1 sinistre mais aucun de chiffrés : ", comparaison["manquant"].sum())

#On regarde aussi le nombre de polices pour lesquelles il manque des sinistres chiffrés par rapport au nombre de polices ayant déclarés des sinistres
intermediaire = differents.loc[differents["nb_lignes_sev"] > 0]
print("Nombre de polices dans le cas intermediaire ou il y a des sinistres déclarés et quelque uns sont chiffrés mais pas la totalité : ", intermediaire.shape[0])

# On peut maintenant comparer les profils des polices pour lesquels il manque des sinistres chiffrés avec ceux pour lesquels il n'en manque pas pour voir si l'on peut identifier un shéma.
profils = freq.merge(comparaison[["IDpol", "manquant"]], on='IDpol')
profils = profils.loc[profils["ClaimNb"] > 0]
print(profils.groupby('manquant')[['Exposure', 'DrivAge', 'BonusMalus']].mean())
print(profils.groupby('manquant').size())

**Incohérence entre les deux tables.** 36 056 sinistres sont déclarés dans `freq` contre 26 639 lignes dans `sev`, soit environ 9600 sinistres sans montant associé, répartis sur 9 116 polices.

Le fait que seulement 9 polices présentent une situation intermédiaire, certains sinistres chiffrés et d'autres non exclut la possibilité qu'une partie massive des sinistres ne soient pas encore chiffrés ou payée.

De plus, après examen des trois variables les plus parlantes sur les profils de ces polices, on constate un écart de moyenne conséquent. En effet la moyenne d'exposition passe de 0.69 à 0.50, celle d'âge de 45 à 49 ans et celle de bonus / malus de 65 à 58.

Cela ne peut pas être du bruit sur plus de 9000 observations, ces différences concernent des conducteurs plus âgés avec de meilleurs coefficients et une durée d'exposition plus courte, il s'agit donc sûrement là d'un segment de portefeuille identifiable.

Pour confirmer cette hypothèse, on peut regarder si les `IDpol` sont segmentés aussi, car il s'agit d'une attribution automatique, et cela montrerait que toutes ces polices ont été extraites en même temps d'une autre base de données et dont les sinistres correspondants n'ont pas été repris.

In [ ]:
print(profils.groupby('manquant')['IDpol'].describe())

L'examen des plages des identifiants le confirme, 75% des polices sans montant ont un `IDpol` inférieur à 20382 et un médian à 10292 alors que les polices sans problèmes sont étalées de 139 à plus de 6 millions.

On peut donc dire que les deux populations occupent des plages d'identifiants pratiquement disjointes, cette différence vient donc d'un problème d'extraction et non d'un phénomène assurantiel.

**Conséquence.** Le modèle de sévérité ne pourra être estimé que sur les sinistres chiffrés, soit environ 74 % du total. Ce sous-ensemble n'est pas représentatif du portefeuille : les polices exclues présentent une exposition moyenne plus courte (0,51 contre 0,69 année), des conducteurs plus âgés (49 contre 45 ans) et de meilleurs coefficients bonus-malus (58 contre 65). Ces variables étant des prédicteurs du modèle, un biais d'estimation est possible.

Les polices concernées disposent d'un `ClaimNb` et d'une `Exposure` valides, elles seront donc conservées dans la base de fréquence car elles apportent une réelle information sur la fréquence. Les exclure reviendrait simplement à effacer une part non négligeable de polices sinistrées tout en conservant l'ensemble des polices non sinistrées.

La restriction s'opérera uniquement et automatiquement lors de la construction de la base de sévérité, en effet une police sans montant de sinistres n'aura simplement rien à apporter à ce modèle. Mais ce constat permet d'être au courant du manque d'une partie non négligeable des données qui auraient été utiles à l'entrainement de ce modèle.

In [ ]:
print((sev["ClaimAmount"] <= 1).sum())
print(sev["ClaimAmount"].quantile([0.01, 0.1, 0.25, 0.5, 0.75, 0.9, 0.99]))
print(sev['ClaimAmount'].value_counts().head(10))

On remarque que certaines valeurs reviennent beaucoup dans les coûts d'un sinistre. Sûrement un montant de provision standard demandé par l'assureur à l'ouverture d'un dossier de sinistre.

Il est très important de le noter car cela aura une incidence sur la suite car ces montants étant vraissemblablement standards et automatiques, il ne dépendent pas à première vue des caractéristiques de l'assuré, c'est un forfait.

### Analyse exploratoire des données

Maintenant que toutes les erreurs sont corrigées et remarques faites sur le dataset, nous pouvons calculer la fréquence annualisée de sinistre.

In [ ]:
print("la fréquence annualisée de sinistres : ", freq['ClaimNb'].sum() / freq['Exposure'].sum())

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].hist(np.log10(sev['ClaimAmount']), bins=120, color='blue')
ax[0].set_xlabel("log10(montant)")
ax[0].set_title("Distribution des montants de sinistres")
ax[1].hist(freq['Exposure'], bins=50, color='lightsteelblue')
ax[1].set_xlabel("Exposition (années-police)")
ax[1].set_title("Distribution de l'exposition")
plt.tight_layout()
plt.show()

Le pic autour de 3,08 correspond au 1204€ que nous avions remarqué plus haut. 

In [ ]:
# Il est intéressant de représenter Density la densité avec une échelle logarithmique pour visualiser les écarts de grandeur.
freq["log_Density"] = np.log(freq["Density"])

In [ ]:

fig, axes = plt.subplots(5, 2, figsize=(13, 23))
plot_freq_par_groupe(freq, "DrivAge", bins=[17,19,21,23,25,30,40,50,60,70,90], ax=axes[0, 0], errorbar=True)
plot_freq_par_groupe(freq, "VehAge", bins=[-1,1,3,5,10,15,30], ax=axes[0, 1])
plot_freq_par_groupe(freq, "BonusMalus", bins=[49,50,55,60,70,80,100,150,230], ax=axes[1, 0])
plot_freq_par_groupe(freq, "log_Density", bins=np.linspace(freq["log_Density"].min(), freq["log_Density"].max(), 12), ax=axes[1, 1], errorbar=True)
plot_freq_par_groupe(freq, "VehPower", bins=None, ax=axes[2, 0], errorbar=True)
plot_freq_par_groupe(freq, "Area", bins=None, ax=axes[2, 1])
plot_freq_par_groupe(freq, "VehGas", bins=None, ax=axes[3, 0])
plot_freq_par_groupe(freq, "VehBrand", bins=None, ax=axes[3, 1], errorbar=True)
plot_freq_par_groupe(freq, "Region", bins=None, ax=axes[4, 0], errorbar=True)

plt.tight_layout()
plt.show()


Chaque graphique croise l'exposition par modalité (axe gauche) et la fréquence des sinistres pondérée par l'exposition (courbe rouge). La ligne pointillée rappelle la fréquence moyenne du portefeuille (0.10).

La lecture conjointe est nécessaire car une fréquence estimée sur une modalité à faible exposition est peu fiable.

**DrivAge.** un des effets les plus marqués du portefeuille. La fréquence atteint 0.28 chez les 18-19 ans, soit près de trois fois la moyenne puis décroît rapidement vers 0.082 vers 30-40 ans. Bien qu'il y ait une assez faible exposition sur les 18 à 21 ans, un facteur 3 sur plus de 1000 sinistres ne s'obtient pas simplement par hasard. On observe aussi un repic sur la tranche 40-50 ans qui, au vu de l'exposition (tranche la plus fournie du portefeuille), n'est pas dû au bruit.

**BonusMalus.** Relation monotone croissante, très forte en amplitude de 0.08 au plancher de Bonus-Malus à 50, à plus de 0.55 au delà de 150, soit 7 fois plus. Ce n'est pas étonnant car le coefficient de bonus-malus comporte directement l'historique des sinistres de l'assuré. Cependant on peut se poser la question de son poids relatif pour l'entrainement du modèle car l'exposition est majoritairement concentrée sur le plancher à 50.

**Density et Area.** Relations monotones croissantes et de forme quasi-identique (traduction de la redondance), la fréquence passe de 0,082 en zone la moins dense à 0,140 en zone la plus dense. Une seule des deux variables sera retenue pour l'entrainement de modèles.

**VehAge** Fréquence élevée sur les véhicules de moins d'un an (0,162), sûrement dû au fait que les véhicules neufs sont mieux assurés et dont les sinistres légers sont systématiquement déclarés. On a ensuite une relation plate entre 1 et 10 ans (0.09), puis une décroissance au delà de 10 ans jusqu'à 0.065.

**VehPower.** Relation très irrégulière, autant dans la fréquence que dans les expositions. Les puissances 12 à 15 ont trop peu d'exposition, un regroupement sera donc nécessaire. Les estimations de la fréquence sur les puissances à peu d'expositions ne sont que très peu précises comme le montrent les intervalles de confiance. On observe un creux marqué à la puissance 8 (0.084) et un maximum autour de 9-10 (0.116).

**VehGas.** Écart faible : 0.098 pour le diesel contre 0.103 pour l'essence, avec des expositions comparables. L'amplitude de l'écart est trop petite par rapport aux écarts avec d'autres variables.

**VehBrand.** Des expositions très disparates ce qui rend les estimations fragiles comme le montre les intervalles de confiance. Un pic sur B12 et un creux sur B14 mais l'anonymisation empèche toute interprétation. Mais de toute façon, l'amplitude de ces fréquences est assez modérée.

**Region.** Forte amplitude (0,073 à 0,141) mais des expositions très déséquilibrées. Les valeurs extrêmes de fréquences tombent systématiquement sur des régions à très faible exposition ce qui rend très incertain l'estimation de la fréquence, comme le confirme l'intervalle de confiance sur celles-ci. Un regroupement des régions à faible exposition est sans doute nécessaire pour l'entrainement des modèles.


In [ ]:
d = freq.copy()
d['age_grp'] = pd.cut(d['DrivAge'], bins=[17,25,30,40,50,60,90])
piv = d.groupby(['age_grp','Area'], observed=True).agg(exposition=('Exposure','sum'), claimnb=('ClaimNb','sum'))
piv['frequence'] = piv['claimnb']/piv['exposition']
piv = piv['frequence'].unstack()

plt.figure(figsize=(8,5))
plt.imshow(piv, aspect='auto', cmap='Reds')
plt.colorbar(label='Fréquence')
plt.xticks(range(len(piv.columns)), piv.columns)
plt.yticks(range(len(piv.index)), [str(i) for i in piv.index])
plt.title("Fréquence par âge et zone")
plt.show()

La heatmap présente la fréquence en fonction de la catégorie d'âge et de la zone d'habitation (A rurale et F très urbaine).

Ce qu'on remarque directement c'est que la zone semble amplifier ou atténuer l'effet de l'âge. Cela se lit facilement sur les 5 dernières lignes où on remarque une croissance de la fréquence quand la densité de la zone croît aussi, avec un pic de croissance quand on passe sur les zones très urbaines E et F.

La fréquence décroît aussi avec l'âge, avec un minimum vers 30-40 ans. 

Cela ne semble cependant pas être le cas pour la catégorie 18-25 ans où la fréquence croît et décroît, de plus en plus fort. Cela peut s'expliquer par le fait que ces catégories avaient nettement moins d'exposition que les autres catégories d'ages, et donc une marge d'erreur plus élevée sur le calcul de la fréquence, d'autant plus si cette faible exposition est en plus découpée en zone.

Mais ce constat ne suffit pas à conclure. En effet un GLM à lien logarithmique suppose que les effets se combinent multiplicativement. La fréquence s'écrirait alors : `base x facteur_âge x facteur_zone`. Mais sous cette hypothèse, le rapport de fréquence entre deux tranches d'âge devrait être le même dans toutes les zones car le facteur zone se simplifie. Est-ce le cas ?

Nous allons donc calculer le rapport de fréquence pour chaque tranche d'âge par rapport à une tranche de référence (40-50 ans) et pour chaque zone, puis comparer.


In [ ]:
# On utilise la ligne 40-50 ans comme ligne de référence.
ratio = piv.div(piv.loc[piv.index[3]], axis=1)
print(ratio.round(2))

En divisant chaque case de chaque ligne par celle correspondante de la ligne de référence, on obtient pour chaque âge son coefficient multiplicatif par rapport à 40-50 ans dans cette zone. Autrement dit, combien de fois une tranche d'âge est plus ou moins risquée dans telle ou telle zone. 

Les lignes ne sont pas constantes. En effet, pour la tranche 30-40 ans, ce coefficient multiplicatif passe de 0.75 en zone A à 1.22 en zone F. Donc avoir 30-40 ans protégerait nettement plus par rapport aux 40-50 ans en zone rurale mais deviendrait un fort désavantage en zone très dense. De plus, le fait que la croissance du coefficient sur cette tranche d'âge soit régulière de zone en zone ne permet pas de l'expliquer par le hasard ou le bruit des données.

Mais la ligne 18-25 ans varie sans ordre apparent, ce qui peut s'expliquer par un bruit d'échantillonnage dû à la faible exposition de la tranche d'âge, ce que nous avions déjà relevé, d'autant plus si elle est découpée en 6 zones.

On peut aussi émettre une réserve sur la zone F qui ne totalise qu'à peu près 8000 années-police, aussi découpées par tranche d'âge. Sa lecture reste donc purement indicative.

**Conséquence pour la modélisation.** Ce type de structure non multiplicative ne pourra pas être représenté par un GLM multiplicatif sans ajout explicite d'interaction. Toutefois, c'est exactement le type de schéma qu'un modèle à base d'arbres capte sans spécification préalable. La question sera creusée lors de la phase 1 par comparaison de déviances avec et sans interaction, puis dans la phase 4 lors de la confrontation au gradient boosting.

## Synthèse de l'exploration

### Corrections d'intégrité appliquées

| Variable | Anomalie constatée | Traitement | Effectif |
|---|---|---|---|
| `Exposure` | Valeurs au dessus de 1, impossibles pour des contrats annuels | Plafonnement à 1 | 1 224 polices (0,18 %) |
| `ClaimNb` | Décroissance anormale des valeurs au dessus de 4 | Plafonnement à 4. | 16 polices (0,002 %) |
| `VehAge` | Décroissance anormale au-dessus de 30, pic à 99 et 100. | Plafonnement à 30 | 1 116 polices |
| `DrivAge` | Décroissance irrégulière au delà de 90, pic à 99. | Plafonnement à 90 | 401 polices |
| `VehGas` | Apostrophes dans les modalités. | Nettoyage de chaîne | Toutes |


Aucune valeur manquante ni doublon dans la table `freq`. Un critère de rejet basé sur la fréquence implicite (`ClaimNb` / `Exposure`) a été testé, or le ratio explosait dès qu'une police avait une exposition courte même avec peu de sinistres, ce qui pénalisait des polices valides tout en gardant des polices à exposition longue avec beaucoup de sinistres. Le plafonnement de `ClaimNb` à 4 est plus simple et plus robuste.


### Caractéristiques du portefeuille

La **Fréquence annualisée** est de 0.1006 sinistres par année polices, soit un sinistre tous les 10 ans pour un contrat annuel. Ce nombre servira de référence pour la suite et pour l'évaluation des modèles. Il est à noter que la fréquence annualisée est calculée sur l'ensemble du portefeuille, y compris les polices sans sinistres et celles avec des sinistres non chiffrés.

### Hiérarchie des variables explicatives.

`BonusMalus` et `DrivAge` sont les variables les plus discriminantes, rapport de 1 à 7 pour BonusMalus (de 0.08 au plancher à 0.55) et de 1 à 3 pour DrivAge (0.082 chez les 30-40 ans à 0.28 chez les 18-19 ans). 

`Area`, `Density` et `VehAge` ont une amplitude moindre, avec un rapport de 1 à 2.

`VehPower`, `VehGas`, `VehBrand` et `Region` ont des effets plus faibles et surtout irréguliers dû à de faibles expositions sur certaines modalités, ce qui rend les estimations peu fiables.

`Area` s'avère être redondante avec `Density`, il faudra donc en choisir une pour l'entrainement des modèles pour ne pas créer de colinéarité dans les GLM.


### Limites du jeu de données

**Incohérence entre les tables.** Nous avons constaté que 36 056 sinistres sont déclarés dans `freq` contre 26 639 lignes dans `sev`, soit environ 9 600 sinistres sans montant associé, répartis sur 9 116 polices. Il s'agit d'un problème d'extraction de données, les polices concernées sont concentrées sur une plage d'identifiants disjointe du reste du portefeuille. Aucune modification n'a été apportée à la table `freq` car ces polices apportent une information sur la fréquence des sinistres, et aucun problème sur `sev` car elles n'y sont donc pas présentes. Cependant cela représente un perte importante pour `sev`. Le modèle de sévérité ne pourra donc être estimé que sur 74% des sinistres, et sur un sous-ensemble de polices dont le profil diffère du portefeuille global (exposition plus courte, conducteurs plus âgés et meilleurs coefficients bonus-malus). Un biais d'estimation est donc possible.

**Montants forfaitaires.** Quatre valeurs concentrent environ 40% des montants de sinistres, il s'agit probablement de montants forfaitaires demandés par l'assureur à l'ouverture d'un dossier de sinistre. Ces montants ne dépendent pas des caractéristiques de l'assuré et ne sont donc pas représentatifs du risque. Cela limitera donc la capacité du modèle de sévérité à discriminer les sinistres selon les caractéristiques des polices. Il faudra donc être vigilant sur l'interprétation des résultats du modèle de sévérité.


**Variables absentes.** Certaines variables comme le kilométrage annuel, l'usage du véhicule, type de contrat, auraient pu être pertinentes pour la modélisation. `Density` et `Area` mesurent la densité dans la zone d'habitation et non sur les routes utilisées par le conducteur.


### Points à trancher en phase 2

- Choix entre `Area`et `Density` (avec `Density` probablement en logarithmique).
- Découpage des variables `DrivAge` et `BonusMalus` en classes pour l'entraînement des modèles.
- Regroupement des modalités à faible exposition pour `VehPower`, `VehBrand` et `Region` pour l'entraînement des modèles.
- Écrêtement des sinistres graves : le 99ᵉ percentile s'établit à 16 800 € pour un
  maximum de 4 millions d'euros. Un sinistre exceptionnel de cet ordre pèse autant que plusieurs
  milliers de sinistres courants et fausserait l'estimation de la sévérité, alors
  que sa survenance n'est pas prédictible à partir des caractéristiques de l'assuré (évênement exceptionnel).
  Un plafonnement suivi d'une réallocation de la charge excédentaire sous forme de
  chargement mutualisé sera étudié.
- Traitement des très petits montants (sinistres à 1€, 1% sous 40€), dont la nature de sinistre réel est douteuse.
- Traitement des sinistres forfaitaires dans le modèle de sévérité.
- Test formel de l'interaction entre `DrivAge` et et la variable géographique retenue (`Density` ou `Area`) pour le modèle de fréquence, et choix du type d'interaction à inclure dans le modèle (si interaction significative).



